<a href="https://colab.research.google.com/github/Muffalo52/anima_lora-Colab-Trainer/blob/dev/anima_lora_trainer_personal_fork.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 🚀 Anima lora 고효율 학습 파이프라인 (Based on : https://github.com/sorryhyun/anima_lora)

# @markdown ### 1. 🔑 인증 설정 (선택)
HF_TOKEN = "" # @param {type:"string"}
WANDB_API_KEY = "" # @param {type:"string"}

# @markdown ### 2. 📁 프로젝트 및 데이터셋 설정
# @markdown `MyDrive/lora_training/datasets/` 경로 내에 있는 폴더명 또는 zip 파일명을 입력하세요.
USE_GOOGLE_DRIVE = True
PROJECT_NAME = "" # @param {type:"string"}

# @markdown ### 3. ⚙️ 학습 환경 설정
MIXED_PRECISION = "fp16" # @param ["bf16", "fp16"]
TARGET_COMMIT = "HEAD" # @param ["HEAD", "3e2c12454eb05bb8f83b5e04cff33852dd7d342f"]
TORCH_COMPILE = True # @param {type:"boolean"}
COMPILE_INDUCTOR_MODE = "default" # @param ["default", "reduce-overhead", "max-autotune"]
# @markdown `IS_T4_GPU`를 체크하면 코랩 무료/기본 GPU(T4) 환경에 맞춰 FlashAttention을 비활성화하고 호환 모드로 안전하게 구동합니다.
IS_T4_GPU = True # @param {type:"boolean"}
GRADIENT_CHECKPOINTING = False # @param {type:"boolean"}
# @markdown 기존에 학습된 LoRA 파일(`.safetensors`)의 경로를 입력하면, 해당 가중치 위에서부터 새로운 학습을 시작합니다. (새로 학습할 경우 비워두세요)
NETWORK_WEIGHTS = "" # @param {type:"string"}
# @markdown 학습 해상도 풀. 띄어쓰기로 구분하세요. (허용값: 512 768 896 1024 1280 1536)
TARGET_RES = "896 1024" # @param {type:"string"}
# # @markdown 기본적으로 제외되는 레이어들을 정규식으로 지정하여 강제로 학습에 포함합니다.
# # @markdown 기본적으로 학습에서 제외되는 시스템 레이어들을 어떻게 처리할지 선택합니다.
INCLUDE_MODE = "\uae30\ubcf8 (\uc81c\uc678 \uc720\uc9c0)"

# @markdown 개별 LoRA 기능 제어[🔗](https://github.com/sorryhyun/anima_lora/tree/main/docs/methods)
USE_ORTHO = False # @param {type:"boolean"}
USE_ORTHO_INIT = False # @param {type:"boolean"}
USE_TIMESTEP_MASK = False # @param {type:"boolean"}
USE_REPA = False # @param {type:"boolean"}
WEIGHT_SVD = True # @param {type:"boolean"}
TRAIN_ADALN = False # @param {type:"boolean"}

# @markdown ### 4. 🎛️ 기본 하이퍼파라미터 튜닝
TRAIN_BATCH_SIZE = 1 # @param {type:"integer"}
GRADIENT_ACCUMULATION_STEPS = 2 # @param {type:"integer"}
# @markdown
LEARNING_RATE = "2e-5" # @param {type:"string"}
NETWORK_DIM = 32 # @param {type:"integer"}
NETWORK_ALPHA = 128 # @param {type:"integer"}
# @markdown
MAX_TRAIN_EPOCHS = 100 # @param {type:"integer"}
SAVE_EVERY_N_EPOCHS = 2 # @param {type:"integer"}
CHECKPOINTING_EPOCHS = 9999 # @param {type:"integer"}

OPTIMIZER_TYPE = "AdamW" # @param ["AdamW", "Prodigy", "ProdigyPlusScheduleFree", "SinkSGD_adv", "AdamCScheduleFreePlus"]
LR_SCHEDULER = "cosine" # @param ["constant", "cosine", "linear"]
LR_WARMUP_RATIO = 0.05 # @param {type:"slider", min:0.0, max:0.2, step:0.01}
# @markdown
TIMESTEP_SAMPLING = "sigmoid" #@param ["uniform", "sigmoid", "shift", "flux_shift"]
SIGMOID_SCALE = 1.2 #@param {type:"number"}
DISCRETE_FLOW_SHIFT = 1.0 # @param {type:"number"}
# @markdown
BLOCKS_TO_SWAP = 0 # @param {type:"integer"}

# @markdown ### 5. 🧠 Prodigy 옵티마이저 전용 설정
prodigy_d_coef = 1.0 # @param {type:"number"}
prodigy_d0 = 1e-6 # @param {type:"number"}
prodigy_schedulefree_c = 0 # @param {type:"number"}

# @markdown ### 6. 📝 캡션 및 텍스트 증강 설정
CAPTION_DROPOUT_RATE = 0.1 # @param {type:"number"}

# @markdown ### 7. 🎭 SAM3 마스킹 (Masked Loss) 설정
# @markdown `USE_MASKED_LOSS`를 켜면, 지정한 프롬프트에 따라 배경이나 노이즈를 학습에서 제외합니다.
USE_MASKED_LOSS = False # @param {type:"boolean"}
MASK_THRESHOLD = 0.5 # @param {type:"number"}
# @markdown 무시할 요소 (예: `text, speech bubble, watermark`). 쉼표로 구분.
MASK_IGNORE_PROMPTS = "" # @param {type:"string"}
# @markdown 집중할 대상 (예: `1girl, character`). 비워두면 무시 요소만 삭제합니다.
MASK_FOCUS_PROMPTS = "" # @param {type:"string"}

# @markdown ### 8. 🥣 Soup (ΔW LoRA Soup) 설정
# @markdown 일반 LoRA 대신 Soup 파이프라인을 사용할지 선택합니다.[🔗](https://github.com/sorryhyun/anima_lora/blob/60c49f0cf097faa2c1b013f93ca6d61def26e618/docs/experimental/soup.md)
USE_SOUP = False # @param {type:"boolean"}
# @markdown 데이터셋 폴더 내에서 학습할 타겟 이미지의 fnmatch 패턴을 지정합니다. (예: 전체 학습시 `*`, 특정 폴더만 학습시 `character_a/*`)
SOUP_PATH_PATTERN = "*" # @param {type:"string"}
# @markdown 1단계(Uncond) 사전학습 에포크 (기본값: 4) 및 샘플링 비율 (1.0 = 100%, 기본값: 1.0). 데이터가 방대할 경우 값을 낮추는 것을 권장합니다.
SOUP_UNCOND_EPOCHS = 4 # @param {type:"integer"}
SOUP_UNCOND_RATIO = 1.0 # @param {type:"number"}
# @markdown 1단계(Uncond) 전용 학습률 (기본값: 2e-5), 네트워크 차원/랭크, 알파 (기본값: dim =32, alpha = 128)
SOUP_UNCOND_LR = "2e-5" # @param {type:"string"}
SOUP_UNCOND_DIM = 32 # @param {type:"integer"}
SOUP_UNCOND_ALPHA = 128 # @param {type:"integer"}

# @markdown ### 9. 🖼️ 검증(Validation) 및 샘플링 설정
# @markdown 검증에 사용할 이미지 장수입니다. (0이면 검증을 수행하지 않습니다)
VALIDATION_SPLIT_NUM = 0 # @param {type:"integer"}
VALIDATE_EVERY_N_EPOCHS = 2 # @param {type:"integer"}
# @markdown CMMD (실제 이미지 생성 품질 평가) 활성화 여부
USE_CMMD = False # @param {type:"boolean"}
VALIDATION_SAMPLE_STEPS = 20 # @param {type:"integer"}
VALIDATION_CFG_SCALE = 1.0 # @param {type:"number"}

import os
import shutil
import subprocess
import re

# 경로 변수 설정
DRIVE_BASE = "/content/drive/MyDrive/lora_training"
DATASET_FOLDER = os.path.join(DRIVE_BASE, "datasets", PROJECT_NAME)
DATASET_ZIP = DATASET_FOLDER + ".zip"
OUTPUT_DIR = os.path.join(DRIVE_BASE, "output", PROJECT_NAME)
LOCAL_DATASET = "/content/anima_lora/image_dataset"

# 1. 드라이브 마운트 및 출력 폴더 생성
if USE_GOOGLE_DRIVE:
    if not os.path.exists('/content/drive'):
        from google.colab import drive
        drive.mount('/content/drive')
    os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. 환경 변수 설정
os.environ['PATH'] = f"/root/.local/bin:{os.environ['PATH']}"
os.environ['PYTHONWARNINGS'] = "ignore"
if HF_TOKEN.strip():
    os.environ['HF_TOKEN'] = HF_TOKEN

# 3 & 4. 환경 구축 및 의존성 설치
print("\n=== [1~2/5] Environment Setup and Dependencies ===")
!apt install -y pigz aria2 -qq
!curl -LsSf https://astral.sh/uv/install.sh | sh
!uv python install 3.13

REPO_URL = "https://github.com/Muffalo52/anima_lora.git"

if not os.path.exists('/content/anima_lora'):
    !git clone {REPO_URL} /content/anima_lora
os.chdir('/content/anima_lora')

if TARGET_COMMIT != "HEAD":
    print(f"체크아웃 진행 중: {TARGET_COMMIT}")
    !git checkout {TARGET_COMMIT}

if os.path.exists('.venv/bin/python'):
    print("Valid virtual environment (.venv) found. Skipping installation.")
else:
    print("Installing environment...")
    !uv venv --python 3.13 --clear
    !uv sync

# 5. 모델 가중치 다운로드
print("\n=== [3/5] Downloading Pre-trained Models ===")
downloads = [
    {"url": "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/vae/qwen_image_vae.safetensors", "dest": "models/vae/qwen_image_vae.safetensors"},
    {"url": "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/text_encoders/qwen_3_06b_base.safetensors", "dest": "models/text_encoders/qwen_3_06b_base.safetensors"},
    {"url": "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/diffusion_models/anima-base-v1.0.safetensors", "dest": "models/diffusion_models/anima-base-v1.0.safetensors"}
]

for item in downloads:
    dest_path = item["dest"]
    url = item["url"]
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)
    if not os.path.exists(dest_path):
        print(f"⬇️ Downloading: {os.path.basename(dest_path)}")
        aria2_cmd = ["aria2c", "--console-log-level=warn", "-c", "-s", "16", "-x", "16", "-k", "10M", "-d", os.path.dirname(dest_path), "-o", os.path.basename(dest_path)]
        if HF_TOKEN.strip():
            aria2_cmd.extend(["--header", f"Authorization: Bearer {HF_TOKEN.strip()}"])
        aria2_cmd.append(url)
        try:
            subprocess.run(aria2_cmd, check=True)
        except subprocess.CalledProcessError as e:
            print(f"\n💥 Error: aria2c download failed for {os.path.basename(dest_path)}. ({e})")
            raise SystemExit("다운로드에 실패하여 프로세스를 종료합니다.")
    else:
        print(f"✅ Already exists: {os.path.basename(dest_path)}")

print("✅ Model download complete.")

# 6. 데이터셋 설정
print("\n=== [4/5] Dataset Preprocessing ===")
if USE_GOOGLE_DRIVE:
    if os.path.exists(LOCAL_DATASET):
        shutil.rmtree(LOCAL_DATASET)
    if os.path.exists(DATASET_FOLDER):
        print(f"Copying folder: {DATASET_FOLDER}")
        shutil.copytree(DATASET_FOLDER, LOCAL_DATASET)
    elif os.path.exists(DATASET_ZIP):
        print(f"Extracting archive: {DATASET_ZIP}")
        shutil.unpack_archive(DATASET_ZIP, LOCAL_DATASET)
    else:
        print("Dataset not found. Skipping preprocessing.")

# 훈련 파라미터 (train_args) 생성
optimizer_args = []
max_grad_norm_cli = ""

if OPTIMIZER_TYPE == "ProdigyPlusScheduleFree":
    OPTIMIZER_TYPE = "prodigyplus.ProdigyPlusScheduleFree"
    optimizer_args = ["betas=0.95,0.99", f"d_coef={prodigy_d_coef}", f"d0={prodigy_d0}"]
    if prodigy_schedulefree_c > 0:
        optimizer_args.append(f"schedulefree_c={prodigy_schedulefree_c}")
    max_grad_norm_cli = "--max_grad_norm 0.0 "
elif OPTIMIZER_TYPE == "Prodigy":
    optimizer_args = [f"d_coef={prodigy_d_coef}", f"d0={prodigy_d0}", "betas=0.9,0.99", "weight_decay=0.01", "decouple=True", "use_bias_correction=True", "safeguard_warmup=True"]
elif OPTIMIZER_TYPE == "AdamW":
    optimizer_args = ["weight_decay=0.1", "betas=0.9,0.99", "fused=True"]
elif OPTIMIZER_TYPE == "SinkSGD_adv":
    OPTIMIZER_TYPE = "adv_optm.optim.SinkSGD_adv.SinkSGD_adv"
    optimizer_args = ["momentum=0.9", "weight_decay=0.01", "cautious_wd=True", "compiled_optimizer=True"]
elif OPTIMIZER_TYPE == "AdamCScheduleFreePlus":
    OPTIMIZER_TYPE = "schedulefree.adamc_schedulefree_plus_paper.AdamCScheduleFreePlusPaper"
    optimizer_args = ["weight_decay=20.0"]


optimizer_args_cli = "--optimizer_args " + " ".join([f'"{arg}"' for arg in optimizer_args]) + " "
compile_args_cli = f"--torch_compile --compile_inductor_mode {COMPILE_INDUCTOR_MODE} " if TORCH_COMPILE else ""
t4_compat_cli = '--attn_mode "torch" ' if IS_T4_GPU else ""
grad_ckpt_cli = "--gradient_checkpointing " if GRADIENT_CHECKPOINTING else ""
unsloth_offload_checkpointing_cli = "--unsloth_offload_checkpointing " if GRADIENT_CHECKPOINTING else ""
save_precision_cli = '--save_precision "fp16" ' if IS_T4_GPU else '--save_precision "bf16" '
network_weights_cli = f'--network_weights "{NETWORK_WEIGHTS.strip()}" ' if NETWORK_WEIGHTS.strip() else ""

network_args_list = []
include_patterns_val = ""
if INCLUDE_MODE == "Modulation 포함":
    include_patterns_val = ".*_modulation.*"
elif INCLUDE_MODE == "전체 포함 (모든 레이어 학습)":
    include_patterns_val = ".*"

if include_patterns_val:
    network_args_list.append(f"include_patterns={include_patterns_val}")

network_args_list.extend([
    f"use_ortho={USE_ORTHO}",
    f"use_ortho_init={USE_ORTHO_INIT}",
    f"use_timestep_mask={USE_TIMESTEP_MASK}",
    f"use_repa={USE_REPA}",
    "min_rank=8",
    "alpha_rank_scale=1.0"
])
network_args_list.append("down_init=weight_svd" if WEIGHT_SVD else "down_init=kaiming")

print(f"✅ LoRA 상세 설정 적용: use_ortho={USE_ORTHO}, use_ortho_init={USE_ORTHO_INIT}, use_timestep_mask={USE_TIMESTEP_MASK}, use_repa={USE_REPA}, weight_svd={WEIGHT_SVD}, train_adaln={TRAIN_ADALN}")

network_args_cli = ""
if network_args_list:
    args_joined = " ".join([f"'{arg}'" for arg in network_args_list])
    network_args_cli = f"--network_args {args_joined} "

wandb_cli = ""
if WANDB_API_KEY.strip():
    wandb_cli = (
        f'--log_with wandb '
        f'--log_tracker_name "Anima_LoRA_Project" '
        f'--wandb_run_name "{PROJECT_NAME}_Run" '
        f'--wandb_api_key "{WANDB_API_KEY.strip()}" '
        f'--log_every_n_steps 1 '
    )

validation_cli = (
    f'--validation_split_num {VALIDATION_SPLIT_NUM} '
    f'--validate_every_n_epochs {VALIDATE_EVERY_N_EPOCHS} '
    f'--validation_sample_steps {VALIDATION_SAMPLE_STEPS} '
    f'--validation_cfg_scale {VALIDATION_CFG_SCALE} '
)
if USE_CMMD:
    validation_cli += '--use_cmmd '
else:
    validation_cli += '--no-use_cmmd '

train_args = (
    f'--output_dir "{OUTPUT_DIR}" '
    f'--output_name "{PROJECT_NAME}" '
    f'--train_batch_size {TRAIN_BATCH_SIZE} '
    f'--network_dim {NETWORK_DIM} '
    f'--network_alpha {NETWORK_ALPHA} '
    f'--learning_rate {LEARNING_RATE} '
    f'--max_train_epochs {MAX_TRAIN_EPOCHS} '
    f'--save_every_n_epochs {SAVE_EVERY_N_EPOCHS} '
    f'--checkpointing_epochs {CHECKPOINTING_EPOCHS} '
    f'--gradient_accumulation_steps {GRADIENT_ACCUMULATION_STEPS} '
    f'{grad_ckpt_cli}'
    f'{unsloth_offload_checkpointing_cli}'
    f'{t4_compat_cli}'
    f'--optimizer_type "{OPTIMIZER_TYPE}" '
    f'{optimizer_args_cli}'
    f'{max_grad_norm_cli}'
    f'{network_args_cli}'
    f'{compile_args_cli}'
    f'--lr_scheduler "{LR_SCHEDULER}" '
    f'--lr_warmup_steps {LR_WARMUP_RATIO} '
    f'--timestep_sampling {TIMESTEP_SAMPLING} '
    f'--sigmoid_scale {SIGMOID_SCALE} '
    f'--discrete_flow_shift {DISCRETE_FLOW_SHIFT} '
    f'--blocks_to_swap {BLOCKS_TO_SWAP} '
    f'--mixed_precision "{MIXED_PRECISION}" '
    f'{save_precision_cli}'
    f'{network_weights_cli}'
    f'--caption_dropout_rate {CAPTION_DROPOUT_RATE} '
    f'--console_log_simple '
    f'{validation_cli}'
    f'{wandb_cli}'
)

# ==============================================================================
# 패치 코드 및 오케스트레이션 정의부
# ==============================================================================
def apply_general_optimizations():
    print("\n[0/3] 인자 및 로깅 제어 패치")

    # 3. base.toml
    base_toml_path = "configs/base.toml"
    if os.path.exists(base_toml_path):
        with open(base_toml_path, "r", encoding="utf-8") as f: toml_content = f.read()
        new_toml_content = re.sub(r'batch_size\s*=\s*\d+', f'batch_size = {TRAIN_BATCH_SIZE}', toml_content)
        new_toml_content = re.sub(r'gradient_accumulation_steps\s*=\s*\d+', f'gradient_accumulation_steps = {GRADIENT_ACCUMULATION_STEPS}', new_toml_content)
        new_toml_content = re.sub(r'validation_split_num\s*=\s*\d+', f'validation_split_num = {VALIDATION_SPLIT_NUM}', new_toml_content)
        if not TORCH_COMPILE:
            new_toml_content = re.sub(r'torch_compile\s*=\s*true', 'torch_compile = false', new_toml_content)
        if USE_MASKED_LOSS:
            if "mask_dir" in new_toml_content:
                new_toml_content = re.sub(r'mask_dir\s*=\s*".*"', 'mask_dir = "post_image_dataset/masks"', new_toml_content)
            else:
                new_toml_content = new_toml_content.replace('[[dataset.subsets]]', '[[dataset.subsets]]\n  mask_dir = "post_image_dataset/masks"')
        new_toml_content = re.sub(r'train_adaln\s*=\s*(true|false|True|False)', f'train_adaln = {str(TRAIN_ADALN).lower()}', new_toml_content)
        if toml_content != new_toml_content:
            with open(base_toml_path, "w", encoding="utf-8") as f: f.write(new_toml_content)
            print("  ✅ [성공] base.toml 제어")
        else:
            print("  ⏭️ [스킵] base.toml 제어 (이미 적용됨)")
)
    # 5. configs/methods/soup.toml 내 use_repa 강제 제어
    soup_toml_path = "configs/soup/soup.toml"
    if os.path.exists(soup_toml_path):
        with open(soup_toml_path, "r", encoding="utf-8") as f: content = f.read()
        new_content = re.sub(r'use_repa\s*=\s*\w+', f'use_repa = {str(USE_REPA).lower()}', content)
        if content != new_content:
            with open(soup_toml_path, "w", encoding="utf-8") as f: f.write(new_content)
            print("  ✅ [성공] soup.toml 내 use_repa 제어")
        else:
            print("  ⏭️ [스킵] soup.toml 내 use_repa 제어 (이미 적용됨)")

    # 6. Uncond 하이퍼파라미터 강제 수정
    soup_base_toml_path = "configs/soup/soup.toml"
    if os.path.exists(soup_base_toml_path):
        with open(soup_base_toml_path, "r", encoding="utf-8") as f: content = f.read()

        # 정규식을 이용해 파일 내부의 기본값을 UI 설정값으로 치환
        content = re.sub(r'learning_rate\s*=\s*[\d\.e\-]+', f'learning_rate = {SOUP_UNCOND_LR}', content)
        content = re.sub(r'network_dim\s*=\s*\d+', f'network_dim = {SOUP_UNCOND_DIM}', content)
        content = re.sub(r'network_alpha\s*=\s*[\d\.]+', f'network_alpha = {SOUP_UNCOND_ALPHA}', content)

        with open(soup_base_toml_path, "w", encoding="utf-8") as f: f.write(content)
        print(f"  ✅ [성공] soup.toml Uncond 전용 파라미터 제어 (LR:{SOUP_UNCOND_LR}, DIM:{SOUP_UNCOND_DIM}, ALPHA:{SOUP_UNCOND_ALPHA})")


def apply_t4_and_fp16_patches():
    import os
    import re
    import glob

    print("\n🚀 T4 및 FP16 최적화 패치\n")

    print("[1/3] flash-attn 제거 및 TOML 설정 제어")
    os.system("uv remove flash-attn > /dev/null 2>&1 || true")
    os.system("uv pip uninstall --python .venv -y flash-attn > /dev/null 2>&1 || true")
    print("  ✅ flash-attn 제거 완료")

    patched_count = 0
    for toml_file in glob.glob("configs/**/*.toml", recursive=True):
        if os.path.exists(toml_file):
            with open(toml_file, "r", encoding="utf-8") as f:
                content = f.read()

            # 1. attn_mode를 flash에서 torch로 변경
            content = re.sub(r'attn_mode\s*=\s*["\']flash["\']', 'attn_mode = "torch"', content)
            # 2. 혼합 정밀도를 T4가 지원하지 않는 bf16에서 fp16으로 변경 (안전 장치)
            content = re.sub(r'mixed_precision\s*=\s*["\']bf16["\']', 'mixed_precision = "fp16"', content)
            # 3. 저장 정밀도를 bf16에서 fp16으로 변경 (안전 장치)
            content = re.sub(r'save_precision\s*=\s*["\']bf16["\']', 'save_precision = "fp16"', content)

            with open(toml_file, "w", encoding="utf-8") as f:
                f.write(content)
            patched_count += 1
    print(f"  ✅ 총 {patched_count}개의 TOML 설정 파일 패치 완료.")

    print("\n[3/3] 정밀도 설정 패치")
    common_path = "scripts/tasks/_common.py"
    if os.path.exists(common_path):
        with open(common_path, "r", encoding="utf-8") as f: code = f.read()

        new_code = re.sub(
            r'(mixed_precision\s*=\s*)["\']bf16["\']',
            rf'\g<1>"{MIXED_PRECISION}"',
            code
        )
        if new_code != code:
            with open(common_path, "w", encoding="utf-8") as f: f.write(new_code)
            print("  ✅ 적용됨: _common.py")
        else:
            if f'"{MIXED_PRECISION}"' in code:
                print("  ⏭️ 스킵 (이미 적용됨): _common.py")
            else:
                print("  ❌ 실패: _common.py에서 'mixed_precision = \"bf16\"' 설정을 찾을 수 없음")

    # train.py 런타임 가로채기 (파라미터 강제 덮어쓰기)
    train_path = "train.py"
    if os.path.exists(train_path):
        with open(train_path, "r", encoding="utf-8") as f:
            train_code = f.read()

        patch_code = """
    # ----------- HOOK -----------
    args.mixed_precision = "fp16"
    args.save_precision = "fp16"

    # args.gradient_checkpointing = True

    # if not hasattr(args, "blocks_to_swap") or getattr(args, "blocks_to_swap") < 24:
    #     args.blocks_to_swap = 12
    # -----------------------------
    trainer.train(args)"""

        # train.py의 마지막 실행 함수인 trainer.train(args)를 찾아서 패치 코드로 바꿔치기
        if "# ----------- HOOK -----------" not in train_code:
            train_code = re.sub(r'trainer\.train\(args\)', patch_code, train_code)
            with open(train_path, "w", encoding="utf-8") as f:
                f.write(train_code)
            print("  ✅ 적용됨: train.py")
        else:
            print("  ⏭️ 스킵 (이미 적용됨): train.py")

    print("\n🚀 패치 스크립트 실행 종료.")


def start_training_pipeline():
    print("\n🛠️ 내부 패치 적용")
    apply_general_optimizations()
    if IS_T4_GPU:
        apply_t4_and_fp16_patches()

    print("\n📦 Downloading Danbooru Tags...")
    !uv run --no-sync python tasks.py download-danbooru-tags

    print("\n🚀 Running image bucketing & VAE latent caching...")
    !uv run --no-sync make preprocess-resize ARGS="--target_res {TARGET_RES}"

    # SAM3 마스킹 파트
    if USE_MASKED_LOSS:
        print("\n🎭 Preparing SAM3 Model & Generating Masks...")
        os.makedirs("models/sam3", exist_ok=True)
        files_to_download = {
            "sam3.pt": "https://huggingface.co/1038lab/sam3/resolve/main/sam3.pt",
            "config.json": "https://huggingface.co/AEmotionStudio/sam3/resolve/main/config.json"
        }
        for filename, url in files_to_download.items():
            dest = os.path.abspath(f"models/sam3/{filename}")
            if not os.path.exists(dest):
                print(f"⬇️ Downloading {filename} via aria2c...")
                aria2_cmd = ["aria2c", "--console-log-level=warn", "-c", "-s", "16", "-x", "16", "-d", "models/sam3", "-o", filename, url]
                subprocess.run(aria2_cmd, check=True)

        ignore_list = [f'"{p.strip()}"' for p in MASK_IGNORE_PROMPTS.split(',') if p.strip()]
        focus_list = [f'"{p.strip()}"' for p in MASK_FOCUS_PROMPTS.split(',') if p.strip()]
        mask_yaml_content = f"prompts: [{ ', '.join(ignore_list) }]\nfocus_prompts: [{ ', '.join(focus_list) }]\nthreshold: {MASK_THRESHOLD}\ndilate: 5\n"
        with open("sam_mask.yaml", "w", encoding="utf-8") as f: f.write(mask_yaml_content)

        target_script = "scripts/preprocess/generate_masks.py"
        if os.path.exists(target_script):
            with open(target_script, "r", encoding="utf-8") as f: script_code = f.read()
            patch_code = """# --- HF HUB OFFLINE INTERCEPT PATCH ---
import os
import huggingface_hub.file_download
_orig_hf_hub_download = huggingface_hub.file_download.hf_hub_download
def _mock_hf_hub_download(repo_id, filename, *args, **kwargs):
    if repo_id == "facebook/sam3":
        if filename == "config.json": return os.path.abspath("models/sam3/config.json")
        if filename.endswith(".pt"): return os.path.abspath("models/sam3/sam3.pt")
    return _orig_hf_hub_download(repo_id, filename, *args, **kwargs)
huggingface_hub.file_download.hf_hub_download = _mock_hf_hub_download
huggingface_hub.hf_hub_download = _mock_hf_hub_download
# --------------------------------------\n"""
            if "# --- HF HUB OFFLINE INTERCEPT PATCH ---" not in script_code:
                with open(target_script, "w", encoding="utf-8") as f: f.write(patch_code + script_code)

        print("🚀 Executing SAM3 generator...")
        !uv run --no-sync python scripts/preprocess/generate_masks.py --config sam_mask.yaml --image-dir post_image_dataset/resized --mask-dir post_image_dataset/masks --recursive

    !uv run --no-sync make preprocess-vae
    !uv run --no-sync make preprocess-te

    if USE_CMMD:
        print("\n🚀 Caching Perceptual Features for CMMD Validation...")
        !uv run --no-sync make preprocess-pe

    if USE_REPA:
        print("\n🚀 Caching REPA Spatial PE features...")
        !uv run --no-sync make preprocess-pe ARGS='--encoder pe_spatial'

    # ------------------------------------------------------------------------------
    # 5. 훈련 시작 분기 파트
    # ------------------------------------------------------------------------------
    import re

    if USE_SOUP:
        print("\n=== [5/5] Starting LoRA Soup Training Pipeline ===")
        print(f"🥣 파이프라인 진행: 무조건부 사전학습(Pool) -> 복수 시드 파인튜닝 -> ΔW SVD 가중치 결합")

        # Soup의 내부 임시파일 저장 시스템을 보호하기 위해, 경로와 이름 덮어쓰기 인자를 임시로 제거합니다.
        soup_train_args = re.sub(r'--output_dir\s+[^\s]+', '', train_args)
        soup_train_args = re.sub(r'--output_name\s+[^\s]+', '', soup_train_args)

        !uv run --no-sync python scripts/soup/pipeline.py \
            --preset default \
            --path_pattern "{SOUP_PATH_PATTERN}" \
            --uncond_epochs {SOUP_UNCOND_EPOCHS} \
            --uncond_ratio {SOUP_UNCOND_RATIO} \
            {soup_train_args}

        # 프로젝트 변수를 활용하여 동적(Dynamic)으로 경로를 지정합니다.
        print(f"\n🚚 완성된 마스터 Soup 모델을 구글 드라이브로 복사합니다...")

        # 1. 만약 해당 프로젝트 이름의 폴더가 없다면 안전하게 새로 만듭니다.
        !mkdir -p {OUTPUT_DIR}

        # 2. 파이썬 shutil과 정규식을 이용하여 임시 파일을 제외한 '진짜 마스터 수프 파일'만 필터링합니다.
        import glob
        import shutil
        import re
        import os

        # 타겟 폴더명에 상관없이 모든 anima_soup 파일을 찾습니다.
        soup_files = glob.glob("/content/anima_lora/output/ckpt/anima_soup_*.safetensors")

        master_files = [f for f in soup_files if not re.search(r'_s\d+\.safetensors', f)]

        if master_files:
            master_soup_file = master_files[0]
            target_path = f"{OUTPUT_DIR}/{PROJECT_NAME}_soup.safetensors"
            shutil.copy2(master_soup_file, target_path)
            print(f"✅ [성공] Soup 모델 저장 완료! 경로: {target_path}")
        else:
            print("❌ [오류] 병합된 최종 Soup 파일을 찾을 수 없습니다.")

    else:
        print("\n=== [5/5] Starting Standard Plain LoRA Model Training ===")
        !uv run --no-sync python train.py --method lora --preset default {train_args}

start_training_pipeline()

In [ ]:
# @title ✔️ SAM3 마스크 테스트
# @markdown SAM3 마스킹 단계까지 완료 후 이 셀을 실행하여 마스크 구역을 확인합니다.
import os
import glob
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

# 검수할 디렉토리 경로 설정
RESIZED_DIR = "post_image_dataset/resized"
MASK_DIR = "post_image_dataset/masks"

# 최대 몇 장까지 출력할지 설정 (0으로 설정 시 제한 없이 모든 이미지 출력)
MAX_DISPLAY_COUNT = 50

# 하위 폴더 내부의 모든 이미지 탐색
image_paths = glob.glob(os.path.join(RESIZED_DIR, "**", "*.*"), recursive=True)
image_paths = [p for p in image_paths if p.lower().endswith(('.png', '.jpg', '.jpeg', '.webp'))]
image_paths.sort()

if not image_paths:
    print(f"❌ {RESIZED_DIR} 폴더 또는 그 하위 폴더 안에 이미지가 존재하지 않습니다.")
elif not os.path.exists(MASK_DIR) or not os.listdir(MASK_DIR):
    print(f"❌ {MASK_DIR} 폴더 안에 마스크 파일이 존재하지 않습니다. 먼저 마스크를 생성해 주세요.")
else:
    display_count = 0

    for img_p in image_paths:
        if MAX_DISPLAY_COUNT > 0 and display_count >= MAX_DISPLAY_COUNT:
            print(f"\n⚠️ 안전을 위해 최대 {MAX_DISPLAY_COUNT}장까지만 출력했습니다. 더 보려면 MAX_DISPLAY_COUNT 값을 늘리세요.")
            break

        rel_path = os.path.relpath(img_p, RESIZED_DIR)
        rel_stem, _ = os.path.splitext(rel_path)

        # 하위 폴더 경로까지 그대로 유지한 채 마스크 파일명 생성
        mask_p = os.path.join(MASK_DIR, f"{rel_stem}_mask.png")
        base_name = os.path.basename(img_p)

        if not os.path.exists(mask_p):
            folder_hint = os.path.dirname(rel_path)
            print(f"⚠️ [{folder_hint}] {base_name}에 해당하는 마스크 파일이 없어 건너뜁니다.")
            continue

        # 이미지 및 마스크 로드
        img = Image.open(img_p).convert("RGB")
        mask = Image.open(mask_p).convert("L")

        # 붉은색 오버레이 레이어 생성
        mask_np = np.array(mask)
        overlay_np = np.zeros((mask_np.shape[0], mask_np.shape[1], 4), dtype=np.uint8)

        # 무시 영역(0)을 반투명한 빨간색으로 채움
        overlay_np[mask_np == 0] = [255, 0, 0, 120]
        overlay_img = Image.fromarray(overlay_np, "RGBA")

        # 원본 이미지 위에 빨간 필터 얹기
        img_rgba = img.convert("RGBA")
        visualized_img = Image.alpha_composite(img_rgba, overlay_img)

        # ----------------------------------------------------
        # 화면 출력
        # ----------------------------------------------------
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))

        axes[0].imshow(img)
        # 제목에 파일이 속한 하위 폴더 표시
        axes[0].set_title(f"Original ({os.path.dirname(rel_path)} / {base_name})", fontsize=10)
        axes[0].axis("off")

        axes[1].imshow(mask, cmap="gray")
        axes[1].set_title("SAM3 Binary Mask\n(White=Train, Black=Ignore)", fontsize=11)
        axes[1].axis("off")

        axes[2].imshow(visualized_img)
        axes[2].set_title("Mask Overlay Result\n(Red Area = Loss Ignored)", fontsize=11, color="red")
        axes[2].axis("off")

        plt.tight_layout()
        plt.show()

        display_count += 1